# Meanflow

我们进入加速生成领域一个独特的部分，一步生成。字面意义上的，我们希望将生成加速到极致，仅仅调用一次模型输出即可生成最终图像。这是奢侈而美丽的。

我们之前所介绍最接近一步生成这一概念的模型是 Consistency Models。CM 通过蒸馏现有 FM 模型学习一个直接的 PF ODE 上任意点到终点的映射，使得极少步生成成为可能。然而，CM 的图像质量还不够使人满意，并且缺乏作为独立模型训练的能力。虽然 CM 存在独立训练方法，但是我们发现其训练是脆弱的，更主流的做法仍是蒸馏现有模型。

Meanflow 的出现则是重大的突破。Meanflow 的原理非常类似原始 Flow Matching，但是做出了一个重要改进，就是驱使模型学习时间维度上平均矢量场而不是瞬时矢量场。更多的，Meanflow 也可以纳入 UCGM 的框架之下。我们详谈。

# 基本原理

正如 Rectified Flow 所指出的，Flow Matching 模型在不经过后训练的情况下，即使使用了最优传输路径作为条件概率路径，其真实 PF ODE 推理轨迹仍是天生弯曲的。这来自于训练时各个条件概率路径的互相干扰。

因此我们提出时间上平均速度矢量场来替代原始 FM 所学习的瞬时速度矢量场。

符号化表述是，假设真实数据点与噪声服从分布 $x \sim p_{data}, \epsilon \sim \mathcal{N}(0, \mathbf{I})$，那么一个条件概率路径可以被写为 $z_t = a_tx + b_t \epsilon$，其中 $a_t, b_t$ 是随时间调度的函数。

更多的，可以写出条件矢量场 $v_t = z_t' = {a_t}' x + {b_t}' \epsilon$，我们记其为 $v_t = v_t(z_t | x)$ 来表示终点为 $x$ 的条件矢量场。

一般的，FM 采取最优传输路径，即传输参数 $a_t = 1-t, b_t = t, v_t = \epsilon - x$。

回顾我们在第五章的结论，根据条件矢量场可以推出边缘矢量场形式 
$$v(z_t,t) = \mathbb{E}_{x \sim p_{data}(x | \hat{z}_t=z_t)} [v_t(\hat{z}_t | x)]$$
此处 $\hat{z}_t$ 仅仅是为了和 $z_t $ 区别，没有特殊含义，仍指的是路径上某点。

同时我们定义条件概率路径 $p_t(\cdot | x) = \mathcal{N}(a_t x, b_t^2 \mathbf{I})$ 以及全局边缘概率路径 
$$p_t(z_t) = \int p_t(z_t |x) p_{data}(x) \mathrm{d}x$$
这表示全体粒子初始遵从高斯噪声分布 $\mathcal{N}(0, \mathbf{I})$ 并且沿着边缘矢量场运动到时间步 $t$ 时遵从的概率分布。

因此上述的边缘矢量场其实还可以写为更加抽象的形式
$$v(z_t, t) = \mathbb{E}_{p_t(v_t|z_t)}[v_t]$$
这指的是对于所有经过 $z_t$ 条件路径在此处对应的条件矢量场 $v_t$。

我不得不说 Meanflow 原文的记号很古怪，并且我指出我们在第五章讲述 FM 时的记号绝对更加清晰。为了统一，我们还是遵从 Meanflow 原文的记号使用，但是如有任何记号上疑问请参照第五章的记号。

现在我们可以定义 FM 的损失函数
$$\mathcal{L}_{\text{FM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1], z_t \sim p_t(z_t)} \|v_\theta(z_t,t) - v(z_t,t)\|^2$$
我们已经证明过，其等价于 
$$\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim \mathcal{U}[0,1], x \sim p_{data}(x), z_t \sim p_t(z_t|x)} \|v_\theta(z_t,t) - v_\theta(z_t|x)\|^2$$

关于推理，则是求解 $$\frac{d z_t}{dt} = v(z_t,t)$$
这需要使用各类求解器逼近 $z_r = z_t - \int_{r}^{t} v(z_\tau , \tau) d\tau $，其中 $z_1 \sim \mathcal{N}(0, \mathbf{I})$。

以上完全是 Flow Matching 原始的内容。接下来我们介绍 Meanflow 的方法。

我们定义一个关于两个时间节点平均的矢量场
$$ u(z_t, r, t) = \frac{1}{t - r} \int_r^t v(z_\tau, \tau) d\tau $$
为了便于区分，我们使用 $u$ 来指代平均矢量场，$v$ 来指代瞬时矢量场。更多的，实际上 $u$ 可以定义为一个关于瞬时速度 $v$ 的泛函结果
$$u = \mathcal{F}(v) = \frac{1}{t - r} \int_r^t v d\tau  $$

同时我们指出，平均矢量场必须满足边界上的一致性条件，即
$$\lim_{r \to t} u =v $$
更加神奇的是，平均矢量场天生地满足一个一致性条件，对于 $s \in [r, t]$
$$(t -r )u(z_t, r, t) = (s- r)u(z_s, r,s) + (t- s)u(z_t, s, t)$$
这是因为积分的可加性
$$ \int_r^t v d\tau = \int_r^s v d\tau + \int_s^t v d\tau$$
因此我们训练 Meanflow 模型完全无需考虑类似 CM 的一致性损失。

更加具魅力的一件事是，如果模型良好地学习平均矢量场的分布，我们可以通过 $u_\theta(\epsilon , 0, 1)$ 直接一步得到预测图像结果。这为一步生成打下了基础。

本质的，平均矢量场是瞬时矢量场路径的捷径，这是因为矢量加法永远遵循从头直接指向尾部的规则。下面这张图展示了这一强大特性。

<img src="./assets/MF.png" width="1000" height="200">

图中可以看到，平均矢量场直接指向了瞬时矢量场的捷径方向，这就意味着更直接更高效的路径模拟。

但是如何学习这个平均矢量场？我为你推导。首先我们写出定义
$$ (t - r) u(z_t, r, t) = \int_r^t v(z_\tau, \tau) d\tau $$
我们将 $r$ 视为常数，对时间步 $t$ 做微分
$$\frac{d}{dt}(t - r)u(z_t, r, t) = \frac{d}{dt} \int_r^t v(z_\tau, \tau) d\tau \implies u(z_t, r, t) + (t - r)\frac{d}{dt}u(z_t, r, t) = v(z_t, t)$$
最后我们得到 
$$\underbrace{u(z_t, r, t)}_{\text{average vel.}} = \underbrace{v(z_t, t)}_{\text{instant. vel.}} - (t - r) \underbrace{\frac{d}{dt} u(z_t, r, t)}_{\text{time derivative}} \quad (*)$$
$(*)$ 式被称为 Meanflow 恒等式。Meanflow 恒等式极其直接地给出了平均速度场的求解方法，但是这里的时间微分还不容易计算。我们继续讨论。

首先我们可以将对于时间步 $t$ 的全微分 $\frac{d}{dt}$ 拆解为对于每一个参数的偏微分

$$\frac{d}{dt}u(z_t, r, t) = \frac{dz_t}{dt}\partial_z u + \frac{dr}{dt}\partial_r u + \frac{dt}{dt}\partial_t u$$

我们知道 $\frac{dz_t}{dt} = v(z_t, t)$，$\frac{dr}{dt} = 0$，以及 $\frac{dt}{dt} = 1$，我们得到了 $u$ 与 $v$ 之间的另一个关系

$${\frac{d}{dt}u(z_t, r, t) = v(z_t, t)\partial_z u + \partial_t u}$$

这意味着全微分实际上是 Jacobina 矩阵 $[ \partial_z u, \partial_r u, \partial_t u]$ 与 $[v, 0, 1]$ 的积。

现在我们可以开始详谈训练。

# 训练与目标函数

我们指出模型需要最小化的损失函数
$$\mathcal{L}_\theta = \mathbb{E} \| u_\theta(z_t,r,t) - \text{sg}(u_{target})\|_2^2 $$
此处的 $u_{target}$ 被称为有效回归目标，形式是我们上述推导的
$$u_{\text{target}} = v(z_t, t) - (t - r) \left( v(z_t, t) \partial_z u_\theta + \partial_t u_\theta \right)$$

请注意 $u_{target}$ 中的 $u_\theta $，这意味着我们无需采样真实矢量场来计算平均矢量场的偏导数而是直接从神经网络中获得，这大大减小了工程实现难度。更多的，停训算子 $\text{sg}$ 防止了模型自己学习自己的崩溃。

更多的，类似我们证明原始 Flow Matching 中的损失 $\mathcal{L}_{FM}$ 等价 $\mathcal{L}_{CFM}$，此处的边缘矢量场对象也可以更换为条件矢量场对象，即 
$$ u_{\text{target}} = v_t - (t - r) \left( v_t \partial_z u_\theta + \partial_t u_\theta \right)$$
在最优传输路径当中，$v_t = \epsilon - x$。

所以，Meanflow 的方法出奇简单，我们仅仅是为 Flow Matching 原始目标函数 MSE 距离加上一个项 $- (t - r) \left( v_t \partial_z u_\theta + \partial_t u_\theta \right)$。实际上如果 $ t=r$，Meanflow 会退化为原始的 Flow Matching。

完整的训练算法是，采样 $\epsilon \sim \mathcal{N}(0, \mathbf{I}), x \sim p_{data}, t, r \sim \mathcal{U}[0,1]$。

计算前向加噪 $z_t = a_t x + b_t \epsilon, v_t = {a_t}' x + {b_t}' \epsilon$。

根据采样结果计算偏微分与模型输出 $u, dudt = \text{jvp}(\theta ,(z_t, r, t), (v_t, 0 ,1))$。

最后计算 $u_{\text{target}} = v_t - (t - r) dudt$ 以及最终损失 $\mathcal{L}_\theta = \mathbb{E} \| u_\theta(z_t,r,t) - \text{sg}(u_{target})\|_2^2 $。

遍历一个 Batch 之后计算损失平均，反向传播更新参数。

请注意算子 $\text{jvp} $，这表示 Jacobian 矩阵与某个位置切向向量的乘积，也就是偏导数的计算。这是前向微分，在 Pytorch 框架的优化下可以与前向传播结果一同产出。虽然偏微分项的产生意味着额外的计算开销，但是原作者统计发现这部分开销不足原始 FM 计算量的 $20\%$，因此这是值得的。

以下是完整训练算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{MeanFlow: Training.} \\
\text{Note: in PyTorch and JAX, } \texttt{jvp} \text{ returns the function output and JVP.} \\
\hline
\\
\color{gray}{\#\ fn(z, r, t)\text{: function to predict } u} \\
\color{gray}{\#\ x\text{: training batch}} \\
\\
t, r = \text{sample\_t\_r}() \\
e = \text{randn\_like}(x) \\
\\
z = (1 - t) \ast x + t \ast e \\
v = e - x \\
\\
u, \text{dudt} = \texttt{jvp}(fn, (z, r, t), (v, 0, 1)) \\
\\
u\_tgt = v - (t - r) \ast \text{dudt} \\
\text{error} = u - \text{stopgrad}(u\_tgt) \\
\\
\text{loss} = \text{metric}(\text{error}) \\
\\
\hline
\end{array}$$

# 推理

推理的核心公式是 $$z_r = z_t - (t-r) u(z_t, r ,t)$$

简单到了极点，推理步长完全由我们决定。甚至一步生成 $$z_0 = z_1 - u(z_1, 0, 1) = \epsilon  - u(z_1, 0, 1)$$
我们不赘述少步推理了。实际上我们可以随意地应用先进的推理采样器。

# 工程技巧

## CFG 内化

我们在上一章讲述 UCGM 就已经提到过 CFG 的巨大弊端，为了向量外推以获得更加明确的图像，模型需要在一次推理中分别调用一次无条件推理与一次有条件推理。因此，我们训练 Meanflow 模型时同样选择将 CFG 内化到训练中的操作。

请注意，我们的符号叙述中，带有条件 $c$ 的项指的是在给定该条件下的情况，而没有条件 $c$ 的项则指的是输入条件 $\emptyset $ 或者对于所有含有条件情况的期望。

我们定义新的原始矢量场 $$v^{\text{cfg}}(z_t, t \mid \mathbf{c}) \triangleq \omega v(z_t, t \mid \mathbf{c}) + (1 - \omega) v(z_t, t)$$
同样的，可以定义新的平均矢量场 $$u^{\text{cfg}}(z_t, r, t \mid \mathbf{c}) = v^{\text{cfg}}(z_t, t \mid \mathbf{c}) - (t - r) \frac{d}{dt} u^{\text{cfg}}(z_t, r, t \mid \mathbf{c})$$
因此可以得到 
$$v^{\text{cfg}}(z_t, t \mid \mathbf{c}) = \omega v(z_t, t \mid \mathbf{c}) + (1 - \omega) u^{\text{cfg}}(z_t, t, t)$$

现在我们给出转化的训练目标函数 $$\mathcal{L}(\theta) = \mathbb{E} \left\| u_\theta^{\text{cfg}}(z_t, r, t \mid \mathbf{c}) - \text{sg}(u_{\text{tgt}}) \right\|_2^2$$
此处 $$u_{\text{tgt}} = \tilde{v}_t - (t - r) \left( \tilde{v}_t \partial_z u_\theta^{\text{cfg}} + \partial_t u_\theta^{\text{cfg}} \right)$$
以及 $$\tilde{v}_t \triangleq \omega v_t + (1 - \omega) u_\theta^{\text{cfg}}(z_t, t, t)$$

在训练中，$10\%$ 的情况是无条件训练。

另外交代一件事，当我们做 CFG 内化，实际上无条件生成的情况是一个不动点，也就是说其和原始矢量场是一致的。这是因为我们认为无条件生成概率分布是有条件生成的期望。换言之
$$v^{\text{cfg}}(z_t, t \mid \emptyset) = v^{\text{cfg}}(z_t, t) \triangleq \mathbb{E}_{\mathbf{c}}[v^{\text{cfg}}(z_t, t \mid \mathbf{c})] = \omega \mathbb{E}_{\mathbf{c}}[v(z_t, t \mid \mathbf{c})] + (1 - \omega) v(z_t, t) = v(z_t, t)$$

## 训练优化

首先是关于损失函数的权重。我们引入一个自适应权重 $$\mathcal{L}_{weighted} = \text{sg}(w) \cdot \|\Delta\|_2^2$$
其中 $w = \frac{1}{ (\|\Delta\|_2^2 + c)^p}$。

设置这一权重是为了防止不稳定的梯度。因为 $$\nabla_\theta \mathcal{L}_{weighted} \approx \frac{1}{\|\Delta\|_2^{2p}} \cdot 2\Delta \cdot \nabla_\theta \Delta \propto \|\Delta\|_2^{1 - 2p} \cdot \nabla_\theta \Delta$$
我们可以通过控制 $p$ 的大小来稳定训练。作者推荐 $p = 0.5$ 或者 $1$。

然后是关于时间步 $r,t$ 的采样问题。我们原始使用 $\mathcal{U}[0, 1]$ 采样，但是实际上完全可以优化。具体来说就是从高斯噪声 $\mathcal{N}(0, \mathbf{I})$ 随机采样两点再映射回到 $(0,1)$，也就是 LogitNorm 采样。同样的，作者推荐后者。

# 总结

Meanflow 的表现非常强劲，676M 参数量的 MeanFlow-XL/2 在 ImageNet $256 \times 256$ 上达到了一步生成 FID $3.43$ 的结果，对比先前是重大的突破。请注意一个震惊的结果，在我们前面介绍的最原始 DDPM 技术中，同参数量模型推理几十步也未必能够达到如此效果。图像生成领域的进步是飞速的。

我在这里提出一个问题：Meanflow 目前最大的痛点是什么？换句话说，Meanflow 的改进工作最应该向哪个方向发展？这个答案非常重要且有趣。你可以先不查看下方文字，思考一会。

现在揭晓答案。我本人初读 Meanflow 时，对于损失函数中的偏微分这一项，有着非常类似 Score Matching 中计算 Hessian 矩阵 Trace 的感觉。他们有着非常大的共同点，额外的神经网络微分计算，不优美的形式。

对于 Score Matching，我们使用 Denoising Score Matching 的方式巧妙避免了这一繁琐计算，现在轮到 Meanflow 了。直觉上也会存在这样一种变换，可以去除此处的偏微分。这将是巨大的进步。

事实就是，我们正向此方向努力。Improved Mean Flows 与 Euler Mean Flows 做出了突破，甚至做到了 JVP-free。下一章我为你介绍 Improved Meanflow 与 Euler Meanflow，前者同样来自何恺明团队。他们做出了符合直觉且优雅的事情。